Below I’ll give you modular PyTorch functions for:

* Real Möbius map ($M_w(x)$) on ball / sphere
* Kuramoto-on-sphere order parameter and vector field
* Reduced hyperbolic ($w$)-dynamics (both via Möbius and via autograd gradient)
* Simple simulation + Plotly visualization hooks for 2D (disk/$S^1$) and 3D (ball/$S^2$)

Everything is real-valued, dot-product based, and autograd-friendly.

# 1. Core math building blocks (PyTorch)

In [1]:
import torch
from torch import Tensor
import math

## 1.1 Basic helpers

In [2]:
def normalize(x: Tensor, eps: float = 1e-9) -> Tensor:
    """
    Normalize vectors along the last dimension.
    x: [..., d]
    returns: [..., d] with unit norm (or zero if input ~0).
    """
    norm = x.norm(dim=-1, keepdim=True)
    return x / (norm + eps)


def dot(a: Tensor, b: Tensor) -> Tensor:
    """
    Dot product along the last dimension.
    a, b: [..., d]
    returns: [...] (scalar per batch entry)
    """
    return (a * b).sum(dim=-1)

## 1.2 Real Möbius map on ball / sphere
This is the real version of the Möbius map ($M_w$) we discussed:

$$
M_w(x) = \frac{(1 - |w|^2)(x - |x|^2 w)}{1 - 2\langle w,x\rangle + |w|^2 |x|^2} - w.
$$

In [3]:
def mobius_ball(x: Tensor, w: Tensor, eps: float = 1e-9) -> Tensor:
    """
    Real Möbius map M_w(x) on the unit ball B^d.

    x: [..., d]   points in R^d, ideally |x| < 1
    w: [..., d] or [d]  parameter vector in B^d (|w| < 1)
       (broadcastable to shape of x)

    returns: [..., d]  = M_w(x)
    """
    # ensure broadcasting
    while w.dim() < x.dim():
        w = w.unsqueeze(0)

    x2 = (x * x).sum(dim=-1, keepdim=True)         # |x|^2
    w2 = (w * w).sum(dim=-1, keepdim=True)         # |w|^2
    wx = (w * x).sum(dim=-1, keepdim=True)         # <w, x>

    num = (1.0 - w2) * (x - x2 * w)
    den = 1.0 - 2.0 * wx + w2 * x2
    den = den.clamp(min=eps)

    return num / den - w


def mobius_sphere(x: Tensor, w: Tensor, eps: float = 1e-9) -> Tensor:
    """
    Möbius map restricted to points on the unit sphere S^{d-1}.
    Uses the simplified formula valid when |x|=1:

    M_w(x) = ((1 - |w|^2)(x - w))/|x - w|^2 - w
    """
    while w.dim() < x.dim():
        w = w.unsqueeze(0)

    w2 = (w * w).sum(dim=-1, keepdim=True)         # |w|^2
    diff = x - w
    diff2 = (diff * diff).sum(dim=-1, keepdim=True)  # |x - w|^2
    diff2 = diff2.clamp(min=eps)

    return (1.0 - w2) * diff / diff2 - w

## 1.3 Hyperbolic metric factor & potential
Hyperbolic metric on the ball ($B^d$):

$$
g_{\text{hyp}} = \phi(w)^2 I, \quad \phi(w) = \frac{2}{1 - |w|^2}.
$$

Relation between Euclidean and hyperbolic gradients:

$$
\nabla_{\text{hyp}}\Phi(w) = \frac{(1 - |w|^2)^2}{4}\nabla\Phi(w).
$$

Hyperbolic potential for linear order parameter:

$$
\Phi(w) = \sum_i a_i \log\frac{|w - p_i|^2}{1 - |w|^2}.
$$

In [4]:
def hyperbolic_conformal_factor(w: Tensor, eps: float = 1e-9) -> Tensor:
    """
    phi(w) = 2 / (1 - |w|^2)
    returns: scalar or [...,1]
    """
    w2 = (w * w).sum(dim=-1, keepdim=True)
    return 2.0 / ((1.0 - w2).clamp(min=eps))


def hyperbolic_potential(
    w: Tensor,
    base_points: Tensor,
    weights: Tensor,
    eps: float = 1e-9
) -> Tensor:
    """
    Phi(w) = sum_i a_i log( |w - p_i|^2 / (1 - |w|^2) )

    w: [d] or [...,d]
    base_points: [N,d]
    weights: [N]
    returns: scalar potential (broadcasted if w has leading dims)
    """
    # make w shape [..., 1, d] to broadcast with [N,d]
    orig_shape = w.shape[:-1]
    d = w.shape[-1]
    w_expanded = w.view(*orig_shape, 1, d)         # [..., 1, d]

    diff = w_expanded - base_points               # [..., N, d]
    num = (diff * diff).sum(dim=-1)               # [..., N]
    num = num.clamp(min=eps)

    w2 = (w * w).sum(dim=-1, keepdim=False)       # [...]
    denom = (1.0 - w2).clamp(min=eps)             # [...]

    # log( |w - p_i|^2 / (1 - |w|^2) )
    log_term = torch.log(num / denom.unsqueeze(-1))   # [..., N]

    # weights [N] -> broadcast to [..., N]
    phi = (log_term * weights.unsqueeze(0)).sum(dim=-1)  # [...]

    return phi

## 1.4 Hyperbolic gradient via autograd
We’ll let autograd handle the Euclidean gradient and then convert to hyperbolic gradient:

In [5]:
def hyperbolic_grad_autograd(
    w: Tensor,
    base_points: Tensor,
    weights: Tensor,
    create_graph: bool = False
) -> Tensor:
    """
    Compute hyperbolic gradient ∇_hyp Phi(w) using autograd.

    w: [d] with requires_grad=True
    base_points: [N,d]
    weights: [N]

    returns: [d]  = ∇_hyp Phi(w)
    """
    assert w.requires_grad, "w must have requires_grad=True"

    phi = hyperbolic_potential(w, base_points, weights)  # scalar
    (grad_euc,) = torch.autograd.grad(
        phi, w, create_graph=create_graph, retain_graph=create_graph
    )

    w2 = (w * w).sum()
    factor = ((1.0 - w2).clamp(min=1e-9) ** 2) / 4.0

    grad_hyp = factor * grad_euc
    return grad_hyp

# 2. Kuramoto / alignment vector fields in real form

## 2.1 Order parameter in transformed coordinates
Given a base configuration ($p_i \in S^{d-1}$), and parameter ($w(t) \in B^d$), the current directions are

$$
x_i(t) = M_w(p_i),
$$

if we ignore the extra rotation ($\zeta_t$) for now.
Then the linear order parameter is

$$
Z(w) = \sum_i a_i x_i(w) = \sum_i a_i M_w(p_i).
$$

In [6]:
def mobius_pushforward_points(
    w: Tensor, base_points: Tensor
) -> Tensor:
    """
    x_i(w) = M_w(p_i) on the sphere (|p_i|=1).

    w: [d]
    base_points: [N,d]
    returns: [N,d]
    """
    return mobius_sphere(base_points, w)


def order_parameter(
    w: Tensor,
    base_points: Tensor,
    weights: Tensor
) -> Tensor:
    """
    Z(w) = sum_i a_i M_w(p_i)

    w: [d]
    base_points: [N,d]
    weights: [N]
    returns: [d]
    """
    x = mobius_pushforward_points(w, base_points)     # [N,d]
    return (weights[:, None] * x).sum(dim=0)          # [d]

## 2.2 Reduced w-dynamics (WS-style, real)
From the L–M–S reduction (with identical A and linear Z), we can use the simple form

$$
\dot w = -\frac12(1 - |w|^2)Z(w), \quad Z(w) = \sum_i a_i M_w(p_i).
$$

In [7]:
def w_vector_field_mobius(
    w: Tensor,
    base_points: Tensor,
    weights: Tensor
) -> Tensor:
    """
    Reduced WS-type vector field for w in B^d:

    dw/dt = -0.5 * (1 - |w|^2) * Z(w),
    with Z(w) = sum_i a_i M_w(p_i).

    w: [d]
    base_points: [N,d]
    weights: [N]

    returns: [d] (dw/dt)
    """
    Z = order_parameter(w, base_points, weights)   # [d]
    w2 = (w * w).sum()
    factor = -0.5 * (1.0 - w2)
    return factor * Z

Alternatively, you can drive ($w$) directly by the negative hyperbolic gradient of ($\Phi$):

In [8]:
def w_vector_field_grad(
    w: Tensor,
    base_points: Tensor,
    weights: Tensor
) -> Tensor:
    """
    Hyperbolic gradient flow:

    dw/dt = -∇_hyp Phi(w)

    (Up to sign conventions; this uses autograd.)
    """
    w = w.clone().requires_grad_(True)
    grad_hyp = hyperbolic_grad_autograd(w, base_points, weights)
    return -grad_hyp.detach()

You can use either `w_vector_field_mobius` (explicit WS formula) or `w_vector_field_grad` (concept-check via autograd) and compare them numerically.

## 2.3 Direct Kuramoto / alignment on the sphere
If you also want the “full” real Kuramoto for directions (without reduction):

$$
\dot x_i = A x_i + Z - \langle Z,x_i\rangle x_i.
$$

In [9]:
def kuramoto_sphere_vector_field(
    x: Tensor,
    A: Tensor,
    weights: Tensor
) -> Tensor:
    """
    Kuramoto / alignment vector field on S^{d-1}:

    x: [N,d]  (assumed |x_i|=1)
    A: [d,d]  skew-symmetric intrinsic rotation
    weights: [N]  alignment weights a_i

    returns: [N,d] = dx/dt
    """
    # x_i in current configuration
    Z = (weights[:, None] * x).sum(dim=0)          # [d]
    proj = (x * Z)                                 # elementwise
    proj = proj.sum(dim=-1, keepdim=True) * x      # <Z,x_i> x_i
    Ax = x @ A.T                                   # [N,d]
    return Ax + Z - proj

# 3. Simple simulation + Plotly visualization hooks
Here’s a tiny Euler integrator for the reduced ($w(t)$) dynamics, plus examples for 2D and 3D visualization.

## 3.1 Simple Euler integration for w(t)

In [10]:
def integrate_w_euler(
    w0: Tensor,
    base_points: Tensor,
    weights: Tensor,
    dt: float,
    steps: int,
    field="mobius"  # or "grad"
):
    """
    Simple Euler integration of w dynamics.

    returns: [steps+1, d] trajectory
    """
    traj = []
    w = w0.clone()
    for _ in range(steps + 1):
        traj.append(w.detach().cpu().clone())
        if field == "mobius":
            dw = w_vector_field_mobius(w, base_points, weights)
        else:
            dw = w_vector_field_grad(w, base_points, weights)
        w = w + dt * dw
        # keep inside ball (to be safe numerically)
        norm = w.norm()
        if norm >= 0.999:
            w = w / (norm + 1e-9) * 0.999
    return torch.stack(traj, dim=0)   # [T,d]

## 3.2 2D case: Kuramoto on S¹ via real embedding
For ($d=2$), you can embed Kuramoto phases ($\theta_k$) as unit vectors:

$$
x_k = (\cos\theta_k,\ \sin\theta_k) \in S^1.
$$

In [11]:
def phases_to_s1_torch(theta: Tensor) -> Tensor:
    """
    theta: [N] phases
    returns: [N,2] points on S^1
    """
    return torch.stack([torch.cos(theta), torch.sin(theta)], dim=-1)

Example usage (no plotting code yet, just data):

In [12]:
# Example: N points on S^1 as base configuration p_i
N = 8
theta0 = torch.linspace(0, 2*math.pi, N+1)[:-1]
p2 = phases_to_s1_torch(theta0)       # [N,2]
weights2 = torch.ones(N)

# start w at origin
w0_2d = torch.zeros(2)

traj_w_2d = integrate_w_euler(
    w0_2d, p2, weights2, dt=0.05, steps=200
)  # [T,2]

# reconstruct directions at some time t*
t_star = -1
x_tstar = mobius_pushforward_points(traj_w_2d[t_star], p2)  # [N,2]

You can now plot:

* `traj_w_2d` as a path in the unit disk,
* `x_tstar` as points on the unit circle (e.g. with Plotly Scatter).

## 3.3 3D case: Kuramoto / alignment on S²
For ($d=3$), you can choose any base configuration on ($S^2$). For example, random points:

In [17]:
def random_points_on_sphere(N: int, d: int = 3) -> Tensor:
    """
    Sample N random unit vectors in R^d.
    """
    x = torch.randn(N, d)
    return normalize(x)


# Example: base configuration p_i on S^2
N = 160
p3 = random_points_on_sphere(N, d=3)   # [N,3]
weights3 = torch.ones(N)

w0_3d = torch.zeros(3)                 # start at hyperbolic origin
traj_w_3d = integrate_w_euler(w0_3d, p3, weights3, dt=0.05, steps=200)

To visualize on the sphere, pick some time index:

In [18]:
t_star = -1
x_tstar_3d = mobius_pushforward_points(traj_w_3d[t_star], p3)  # [N,3]

Now you can plug this into a Plotly figure similar to your example:

In [16]:
import numpy as np
import plotly.graph_objects as go

def plot_s2_points(points: np.ndarray, title="S2 configuration"):
    """
    points: [N,3] numpy array on S^2
    """
    # sphere mesh
    u = np.linspace(0, 2*np.pi, 64)
    v = np.linspace(0, np.pi, 32)
    U, V = np.meshgrid(u, v)
    X = np.cos(U) * np.sin(V)
    Y = np.sin(U) * np.sin(V)
    Z = np.cos(V)

    fig = go.Figure()
    fig.add_trace(go.Surface(
        x=X, y=Y, z=Z,
        opacity=0.2, showscale=False,
        colorscale="Greys"
    ))
    fig.add_trace(go.Scatter3d(
        x=points[:,0], y=points[:,1], z=points[:,2],
        mode="markers",
        marker=dict(size=4, color="red"),
        name="Oscillators"
    ))
    fig.update_layout(
        title=title,
        scene=dict(
            aspectmode="data",
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
        ),
        margin=dict(l=0, r=0, t=40, b=0)
    )
    return fig

# Example usage:
fig_3d = plot_s2_points(x_tstar_3d.detach().cpu().numpy(),
                        title="Kuramoto directions on S^2 at t*")
fig_3d.show()

You could similarly make a `plot_disk_2d(traj_w_2d)` to watch the hyperbolic parameter ($w(t)$) move around inside the unit disk.

## How this all fits your goal
**Modularity:**

Every mathematical object (Möbius map, order parameter, potential, hyperbolic gradient, Kuramoto vector field) is its own standalone function on torch tensors.

**Real-valued / matrix-friendly:**

Everything is written with dot products, broadcasting, and real tensors – no complex numbers.

**Autograd-ready:**

* `hyperbolic_potential` is a pure tensor function.
* `hyperbolic_grad_autograd` uses torch.autograd.grad to get ($\nabla_{\text{hyp}}\Phi$), which you can reuse to test or optimize parameters.

**Visualizations:**
* 2D: use `traj_w_2d` in the disk + `mobius_pushforward_points` to see how base phases on the circle move.
* 3D: use `traj_w_3d` + Plotly sphere plotting to see the evolving configuration on ($S^2$).

If you’d like, next step could be:

* wrapping this into a small nn.Module that learns weights ($a_i$) to fit observed velocity-direction data,
* or adding intrinsic rotations ($A$) and visualizing the combined effect in these hyperbolic coordinates.